# <center>Sentiment Analysis on IMDB Movie Reviews using an RNN</center>
---
This notebook builds an end-to-end **binary sentiment classifier** (positive vs. negative) for the IMDB movie review dataset.

**Pipeline overview:**
1. Load and inspect the raw dataset
2. Clean and normalize the review text
3. Encode labels and vectorize text with TF-IDF
4. Split into train/test sets and wrap them in PyTorch `DataLoader`s
5. Build and train a Recurrent Neural Network (RNN) classifier
6. Evaluate the trained model on held-out test data

Every code cell is preceded by a short explanation of what it does and why, and inline comments describe individual steps.

## 01. Load the Dataset

In [2]:
import pandas as pd

reviews_df = pd.read_csv("IMDB Dataset.csv")

### Inspecting the Dataset

In [3]:
reviews_df.shape

(50000, 2)

In [4]:
reviews_df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
reviews_df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
reviews_df.duplicated().sum()

np.int64(418)

In [7]:
reviews_df.drop_duplicates(inplace=True)
reviews_df.shape

(49582, 2)

## 02. Text Preprocessing

### 2.1 Convert to lowercase

In [8]:
reviews_df["review"] = reviews_df["review"].str.lower()

### 2.2 Removing URLs

In [9]:
import re

def remove_urls(text):
    """Strip http(s) URLs from a string."""
    return re.sub(r"http\S+", "", text)

reviews_df["review"] = reviews_df["review"].apply(remove_urls)

### 2.3 Removing Punctuation

In [10]:
def remove_punctuation(text):
    """Keep only alphanumeric characters and whitespace."""
    return re.sub(r"[^A-Za-z0-9\s]", "", text)

reviews_df["review"] = reviews_df["review"].apply(remove_punctuation)

In [11]:
reviews_df.head()  

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 2.4 Removing HTML Tags

In [12]:
def remove_html_tags(text):
    """Strip HTML tags such as <br />."""
    return re.sub(r"<.*?>", "", text)

reviews_df["review"] = reviews_df["review"].apply(remove_html_tags)

### 2.5 Removing Stopwords

In [13]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Error loading punkt: <urlopen error pathsec.urlopen: no
[nltk_data]     validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
[nltk_data] Error loading punkt_tab: <urlopen error pathsec.urlopen:
[nltk_data]     no validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
[nltk_data] Error loading stopwords: <urlopen error pathsec.urlopen:
[nltk_data]     no validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>


False

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english")) 

In [15]:
def remove_stopwords(text):
    """Tokenize the text and keep only tokens that are not English stopwords."""
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token not in english_stopwords]
    return " ".join(filtered_tokens)

reviews_df["review"] = reviews_df["review"].apply(remove_stopwords)

In [16]:
reviews_df.head() 

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


### 2.6 Stemming

In [17]:
from nltk.stem import PorterStemmer

porter_stemmer = PorterStemmer()

def stem_text(text):
    """Stem every token in the text and rejoin into a single string."""
    tokens = word_tokenize(text)
    stemmed_tokens = [porter_stemmer.stem(token) for token in tokens]
    return " ".join(stemmed_tokens)

reviews_df["review"] = reviews_df["review"].apply(stem_text)

### 2.7 Encoding the Labels

In [18]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
reviews_df["sentiment"] = label_encoder.fit_transform(reviews_df["sentiment"])

labels = reviews_df["sentiment"]
labels

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 2.8 Vectorizing the Text with TF-IDF

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
feature_matrix = tfidf_vectorizer.fit_transform(reviews_df["review"])  # sparse (n_reviews, 5000) matrix

## 3. Datasets and Data Loaders

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    feature_matrix, labels, test_size=0.2, random_state=42
)

In [21]:
X_train.shape

(39665, 5000)

In [22]:
X_test.shape

(9917, 5000)

- `feature_matrix` is a sparse matrix (most TF-IDF entries are zero). PyTorch tensors need dense arrays, so we convert the train/test splits to dense NumPy arrays before building tensors.

In [23]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

- `TensorDataset` pairs each feature vector with its label so a `DataLoader` can iterate over `(review_features, label)` pairs. We cast everything to `float32` tensors since that's the dtype PyTorch layers expect by default.

In [24]:
train_dataset = TensorDataset(
    torch.from_numpy(X_train_dense).float(),
    torch.from_numpy(y_train.values).float()
)

test_dataset = TensorDataset(
    torch.from_numpy(X_test_dense).float(),
    torch.from_numpy(y_test.values).float()
)

- `DataLoader` batches the dataset and (for training) shuffles it each epoch so the model doesn't learn anything from the order the reviews happen to be in.

In [25]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## 4. Building the RNN
We treat each review's 5,000-dimensional TF-IDF vector as a **single-timestep sequence** fed into a recurrent layer, followed by a linear layer that outputs one logit for binary classification.

In [26]:
import torch.nn as nn
import torch.optim as optim

In [27]:
class SentimentRNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

### Instantiating the Model, Loss, and Optimizer

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = X_train_dense.shape[1]  
model = SentimentRNN(input_size).to(device)

criterion = nn.BCELoss()                      
optimizer = optim.Adam(model.parameters())   

## 5. Training the RNN

In [29]:
num_epochs = 8

for epoch in range(num_epochs):
    model.train()

    for review_batch, label_batch in train_loader:
        review_batch = review_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad()

        review_batch = review_batch.unsqueeze(1)         
        logits = model(review_batch)                      
        predicted_probs = torch.sigmoid(logits.squeeze()) 

        loss = criterion(predicted_probs, label_batch) 
        loss.backward()                                 
        optimizer.step()                                 
    print(f"Epoch {epoch + 1}/{num_epochs} - Loss: {loss.item():.4f}")

Epoch 1/8 - Loss: 0.2276
Epoch 2/8 - Loss: 0.2410
Epoch 3/8 - Loss: 0.1830
Epoch 4/8 - Loss: 0.1605
Epoch 5/8 - Loss: 0.0984
Epoch 6/8 - Loss: 0.2231
Epoch 7/8 - Loss: 0.1461
Epoch 8/8 - Loss: 0.1537


## 6. Evaluating the Model


In [33]:
model.eval()

correct_predictions = 0
total_predictions = 0

with torch.no_grad():  
    for review_batch, label_batch in test_loader:
        review_batch = review_batch.to(device).unsqueeze(1)
        label_batch = label_batch.to(device)

        logits = model(review_batch)
        predicted_probs = torch.sigmoid(logits.squeeze())
        predicted_labels = (predicted_probs > 0.5).float()  

        total_predictions += label_batch.size(0)
        correct_predictions += (predicted_labels == label_batch).sum().item()

accuracy = correct_predictions / total_predictions * 100
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 87.30%


In [34]:
torch.save(model.state_dict(), 'rnn_sentiment.pth')
print("Model saved successfully as 'rnn_sentiment.pth'!")

Model saved successfully as 'rnn_sentiment.pth'!


In [32]:
import pickle

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Saved tfidf_vectorizer.pkl and label_encoder.pkl")
print("Now copy these 3 files into sentiment-app/backend/artifacts/:")
print("  - rnn_sentiment.pth")
print("  - tfidf_vectorizer.pkl")
print("  - label_encoder.pkl")

Saved tfidf_vectorizer.pkl and label_encoder.pkl
Now copy these 3 files into sentiment-app/backend/artifacts/:
  - rnn_sentiment.pth
  - tfidf_vectorizer.pkl
  - label_encoder.pkl
